# 📘 Módulo 5 — Teste de Hipóteses
## Livro Didático Aplicado (Híbrido)

- 🔵 **Conteúdo oficial do módulo 5 (IBM)**
- 🟣 **Conteúdo expandido (Livro Didático)**
- 🟠 **Conteúdo avançado (Opcional, matemático)**

Este notebook segue o mesmo padrão dos módulos anteriores:

- clareza didática,
- profundidade conceitual,
- rigor matemático,
- explicações intuitivas,
- aplicações práticas,
- demonstrações opcionais em `<details>`.

Você pode:
- seguir só o que é 🔵 (curso),
- explorar o 🟣 (livro didático),
- abrir o 🟠 (avançado) quando quiser ir mais fundo.

# 📚 Índice

0. [Setup — bibliotecas e dados](#setup)
1. [Introdução ao Teste de Hipóteses](#intro)
2. [Teste Z vs Teste T](#zvt)
3. [Caudas e regiões de rejeição](#caudas)
4. [Teste T para duas amostras independentes](#t2samples)
5. [Variâncias iguais vs desiguais (Levene)](#levene)
6. [ANOVA — Análise de Variância](#anova)
7. [Testes de correlação (Qui‑quadrado e Pearson)](#correlacao)
8. [Aplicações com teaching ratings](#aplicacoes)
9. [Exercícios guiados](#exercicios)
10. [Apêndice matemático avançado](#apendice)

<a id="setup"></a>
# 0. Setup — bibliotecas e dados

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm, t, levene, f_oneway, chi2_contingency, pearsonr

plt.style.use("seaborn-v0_8-whitegrid")

🔵 **Dataset teaching ratings (IBM)**

O módulo 5 continua usando o mesmo dataset dos módulos anteriores.
Ele contém:
- avaliações de ensino (`eval`),
- características dos instrutores,
- número de alunos,
- variáveis categóricas e numéricas.

Este dataset será usado para:
- testes t,
- testes z,
- ANOVA,
- testes de correlação,
- testes qui‑quadrado.

In [ ]:
ratings_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ST0151EN-SkillsNetwork/labs/teachingratings.csv"
ratings_df = pd.read_csv(ratings_url)
ratings_df.to_csv("/home/moacir/projects/ml/IBM/statistics/data/raw/teachingratings.csv", index=False)
ratings_df.head()

In [ ]:
# Carregamento local
ratings_df = pd.read_csv("/home/moacir/projects/ml/IBM/statistics/data/raw/teachingratings.csv")
ratings_df.head()

---
# 🔵 Conexão com o curso IBM

O módulo 5 apresenta:

1. **Teste Z vs Teste T**  
   - quando usar cada um  
   - desvio padrão populacional conhecido vs desconhecido  
   - quatro cenários clássicos  

2. **Caudas e regiões de rejeição**  
   - valores críticos (1.96, 1.64)  
   - testes bilaterais e unilaterais  

3. **Variâncias iguais vs desiguais**  
   - teste de Levene  
   - variância combinada  
   - regra prática da razão < 1.5  

4. **ANOVA (One-way ANOVA)**  
   - comparação de 3+ médias  
   - distribuição F  

5. **Testes de correlação**  
   - Qui‑quadrado para variáveis categóricas  
   - Pearson para variáveis contínuas  

Este notebook segue exatamente essa ordem — com explicações ampliadas, rigor matemático e visualizações adicionais.

---
# 🟣 Antes de avançar

A partir daqui, entraremos no conteúdo central do módulo:

- hipóteses nula e alternativa,  
- testes z e t,  
- regiões críticas,  
- variâncias iguais vs desiguais,  
- ANOVA,  
- correlação,  
- qui‑quadrado.

Cada seção terá:

- definição clara,  
- explicação intuitiva,  
- interpretação geométrica,  
- exemplos numéricos,  
- gráficos,  
- e aprofundamentos opcionais em `<details>`.

Vamos começar pela base: **o que é um teste de hipótese**.

<a id="intro"></a>
# 1. Introdução ao Teste de Hipóteses

🔵 **Conteúdo oficial do módulo 5 (IBM)**

O curso inicia o módulo 5 explicando que testes de hipóteses são usados para:

- comparar médias,
- comparar proporções,
- comparar variâncias,
- verificar associações entre variáveis,
- avaliar correlação.

O foco inicial é:
- **comparação de médias** usando testes Z e T.

Vamos construir a base conceitual antes de entrar nos testes específicos.

## 1.1 O que é um teste de hipótese?

Um **teste de hipótese** é um procedimento estatístico para avaliar uma afirmação
sobre um parâmetro populacional.

Os elementos fundamentais são:

### 🔹 Hipótese nula (H₀)
A afirmação que assumimos como verdadeira até que haja evidência suficiente contra ela.

Exemplo:
$$
H_0: \mu_1 = \mu_2
$$

### 🔹 Hipótese alternativa (Hₐ)
A afirmação que queremos investigar.

Exemplo:
$$
H_a: \mu_1 \ne \mu_2
$$

### 🔹 Estatística de teste
Uma medida calculada a partir dos dados que indica quão distante estamos do valor esperado sob H₀.

### 🔹 Valor‑p
Probabilidade de observarmos um valor tão extremo quanto o observado **assumindo que H₀ é verdadeira**.

### 🔹 Nível de significância (α)
Probabilidade máxima de rejeitar H₀ quando ela é verdadeira (erro tipo I).

Valores típicos:
- 0.05  
- 0.01  
- 0.10  

## 1.2 Estrutura geral de um teste de hipótese

Todo teste segue os mesmos passos:

1. **Declare H₀ e Hₐ**
2. **Escolha o teste apropriado** (Z, T, ANOVA, Qui‑quadrado…)
3. **Calcule a estatística de teste**
4. **Calcule o valor‑p**
5. **Compare com α**
6. **Decida**: rejeitar ou não rejeitar H₀

O módulo 5 da IBM foca principalmente nos testes:

- **Z‑test**  
- **T‑test**  
- **ANOVA**  
- **Qui‑quadrado**  
- **Correlação de Pearson**

## 1.3 Tipos de testes de hipótese

Existem três formas principais de hipóteses alternativas:

### 🔹 Teste bilateral (two‑tailed)
$$
H_a: \mu_1 \ne \mu_2
$$

### 🔹 Teste unilateral à direita (right‑tailed)
$$
H_a: \mu_1 > \mu_2
$$

### 🔹 Teste unilateral à esquerda (left‑tailed)
$$
H_a: \mu_1 < \mu_2
$$

A escolha depende da pergunta de pesquisa.

## 1.4 Interpretação geométrica

A decisão de rejeitar ou não H₀ depende de onde a estatística de teste cai
na distribuição teórica (normal ou t).

Vamos visualizar isso.

In [ ]:
x = np.linspace(-4, 4, 400)
y = norm.pdf(x)

plt.figure(figsize=(8, 3))
plt.plot(x, y, color="steelblue")

# regiões críticas para α = 0.05 (teste bilateral)
alpha = 0.05
z_crit = norm.ppf(1 - alpha/2)

plt.fill_between(x, y, where=(x < -z_crit), color="red", alpha=0.3)
plt.fill_between(x, y, where=(x >  z_crit), color="red", alpha=0.3)

plt.title("Regiões de rejeição — Teste bilateral (α = 0.05)")
plt.xlabel("z")
plt.ylabel("densidade")
plt.grid(alpha=0.3)
plt.show()

🟣 **Interpretação**

- As áreas vermelhas representam regiões onde rejeitamos H₀.
- A área total dessas regiões é α = 0.05.
- Se a estatística de teste cair nessas regiões, rejeitamos H₀.

Isso vale tanto para testes Z quanto para testes T.

<details>
<summary>🟠 Explicação avançada — Erros tipo I e tipo II</summary>

Em testes de hipótese, existem dois tipos de erro:

### 🔹 Erro tipo I (α)
Rejeitar H₀ quando ela é verdadeira.

### 🔹 Erro tipo II (β)
Não rejeitar H₀ quando ela é falsa.

A **potência** do teste é:

$$
1 - \beta
$$

Quanto maior a potência, maior a capacidade de detectar diferenças reais.

</details>

---
## 1.5 Exemplo simples (conceitual)

Suponha que queremos testar se a média de avaliações de instrutores do sexo masculino
é igual à média de instrutores do sexo feminino.

Hipóteses:

$$
H_0: \mu_{\text{male}} = \mu_{\text{female}}
$$

$$
H_a: \mu_{\text{male}} \ne \mu_{\text{female}}
$$

Este é um **teste bilateral**.

No módulo 5, aprenderemos a:
- escolher entre Z e T,
- calcular a estatística,
- calcular o valor‑p,
- interpretar o resultado.

<a id="zvt"></a>
# 2. Teste Z vs Teste T

🔵 **Conteúdo oficial do módulo 5 (IBM)**

O curso explica que:

> “Nos testes de hipóteses tradicionais, temos a opção de realizar o teste Z ou o teste T.  
> Se o desvio padrão populacional é conhecido → usamos Z.  
> Se o desvio padrão populacional é desconhecido → usamos T.”  

E apresenta **quatro cenários clássicos**.

Vamos organizar tudo de forma clara e aprofundada.

## 2.1 Quando usar o Teste Z?

Use **Z‑test** quando:

- o **desvio padrão populacional (σ)** é conhecido  
- a amostra é grande (regra prática: n ≥ 30)  
- a distribuição é aproximadamente normal  

Cenário típico:

$$
H_0: \mu = \mu_0
$$

Estatística:

$$
Z = \frac{\bar{X} - \mu_0}{\sigma / \sqrt{n}}
$$

## 2.2 Quando usar o Teste T?

Use **T‑test** quando:

- o **desvio padrão populacional é desconhecido**  
- usamos o **desvio padrão amostral (s)**  
- a amostra é pequena (n < 30)  

Estatística:

$$
T = \frac{\bar{X} - \mu_0}{s / \sqrt{n}}
$$

A distribuição usada é a **t de Student**, com:

$$
\nu = n - 1
$$

## 2.3 Os quatro cenários clássicos (IBM)

O módulo 5 apresenta exatamente estes quatro casos:

### 🔹 Cenário 1 — Média da amostra vs média populacional (σ conhecido)
→ **Teste Z**

### 🔹 Cenário 2 — Média da amostra vs média populacional (σ desconhecido)
→ **Teste T**

### 🔹 Cenário 3 — Duas amostras independentes, variâncias desiguais
→ **Teste T** (Welch)

### 🔹 Cenário 4 — Duas amostras independentes, variâncias iguais
→ **Teste T** (pooled)

🔵 Isso está exatamente no conteúdo IBM que você anexou.

## 2.4 Valores críticos: 1.96 e 1.64

O curso enfatiza:

- **1.96** → teste bilateral (α = 0.05)  
- **1.64** → teste unilateral (α = 0.05)  

Esses valores vêm da normal padrão:

$$
P(|Z| > 1.96) = 0.05
$$

$$
P(Z > 1.64) = 0.05
$$

In [ ]:
# Visualização dos valores críticos
x = np.linspace(-4, 4, 400)
y = norm.pdf(x)

plt.figure(figsize=(8, 3))
plt.plot(x, y, color="steelblue")

# bilateral 1.96
plt.fill_between(x, y, where=(x < -1.96), color="red", alpha=0.3)
plt.fill_between(x, y, where=(x >  1.96), color="red", alpha=0.3)

# unilateral 1.64
plt.fill_between(x, y, where=(x > 1.64), color="orange", alpha=0.3)

plt.axvline(1.96, color="black", linestyle="--")
plt.axvline(-1.96, color="black", linestyle="--")
plt.axvline(1.64, color="black", linestyle="--")

plt.title("Valores críticos: 1.96 (bilateral) e 1.64 (unilateral)")
plt.xlabel("z")
plt.ylabel("densidade")
plt.grid(alpha=0.3)
plt.show()

🟣 **Interpretação**

- As áreas vermelhas representam a região crítica do teste bilateral.  
- A área laranja representa a região crítica do teste unilateral.  
- Se a estatística de teste cair nessas regiões → rejeitamos H₀.

## 2.5 Exemplo IBM — Teste bilateral

O curso diz:

> “Se a estatística Z ou T for maior que o valor absoluto de 1.96, rejeitamos H₀.”

Vamos ilustrar com um exemplo simples.

In [ ]:
z_obs = 2.3
p_value = 2 * (1 - norm.cdf(abs(z_obs)))
z_obs, p_value

🟣 **Interpretação**

- |2.3| > 1.96 → cai na região crítica  
- valor‑p < 0.05 → rejeitamos H₀  

Isso está exatamente alinhado com o conteúdo IBM.

## 2.6 Exemplo IBM — Teste unilateral

O curso diz:

> “Se a estatística Z ou T for maior que 1.64, rejeitamos H₀ no teste unilateral.”

Vamos ilustrar.

In [ ]:
z_obs = 1.8
p_value = 1 - norm.cdf(z_obs)
z_obs, p_value

🟣 **Interpretação**

- 1.8 > 1.64 → rejeitamos H₀  
- valor‑p < 0.05  

Isso corresponde exatamente ao vídeo do módulo 5.

<details>
<summary>🟠 Explicação avançada — Por que Z e T são tão parecidos?</summary>

Quando o desvio padrão populacional é desconhecido, usamos o desvio padrão amostral $s$.

Isso introduz variabilidade extra:

$$
T = \frac{Z}{\sqrt{V/\nu}}
$$

onde $V \sim \chi^2_\nu$.

Para amostras grandes:

$$
\chi^2_\nu / \nu \to 1
$$

então:

$$
T \to Z
$$

Por isso, para n ≥ 30, Z e T são praticamente iguais.

</details>

<a id="caudas"></a>
# 3. Caudas e Regiões de Rejeição

🔵 **Conteúdo oficial do módulo 5 (IBM)**

O curso explica que:

> “Para um teste bicaudal, usamos 1.96 como limite para rejeitar H₀.  
> Para um teste unilateral, usamos 1.64.”  

E detalha como:
- dividir α entre as caudas,
- interpretar regiões de rejeição,
- relacionar hipóteses com caudas,
- visualizar a curva normal ou t.

Vamos organizar isso de forma clara, visual e aprofundada.

## 3.1 Revisão: hipóteses e direção do teste

A forma da hipótese alternativa determina **qual cauda** será usada.

### 🔹 Teste bilateral
$$
H_a: \mu_1 \ne \mu_2
$$
→ duas caudas  

### 🔹 Teste unilateral à esquerda
$$
H_a: \mu_1 < \mu_2
$$
→ cauda esquerda  

### 🔹 Teste unilateral à direita
$$
H_a: \mu_1 > \mu_2
$$
→ cauda direita  

## 3.2 Teste bilateral (two‑tailed)

🔵 **Conteúdo IBM**

> “Para testes bicaudais, 5% da área é dividida igualmente:  
> 2.5% na cauda esquerda e 2.5% na cauda direita.”

Isso corresponde ao valor crítico:

$$
|z| > 1.96
$$

Vamos visualizar.

In [ ]:
x = np.linspace(-4, 4, 400)
y = norm.pdf(x)

plt.figure(figsize=(8, 3))
plt.plot(x, y, color="steelblue")

# regiões críticas
plt.fill_between(x, y, where=(x < -1.96), color="red", alpha=0.35)
plt.fill_between(x, y, where=(x >  1.96), color="red", alpha=0.35)

plt.axvline(-1.96, color="black", linestyle="--")
plt.axvline(1.96, color="black", linestyle="--")

plt.title("Região de rejeição — Teste bilateral (α = 0.05)")
plt.xlabel("z")
plt.ylabel("densidade")
plt.grid(alpha=0.3)
plt.show()

🟣 **Interpretação**

- As áreas vermelhas somam 5% da curva.  
- Se a estatística de teste cair nessas regiões → rejeitamos H₀.  
- Isso vale tanto para Z quanto para T (com df apropriado).  

## 3.3 Teste unilateral à esquerda (left‑tailed)

🔵 **Conteúdo IBM**

> “Se a hipótese alternativa diz que a diferença é menor que zero,  
> toda a região de rejeição está na cauda esquerda.”

Valor crítico:

$$
z < -1.64
$$

In [ ]:
plt.figure(figsize=(8, 3))
plt.plot(x, y, color="steelblue")

plt.fill_between(x, y, where=(x < -1.64), color="red", alpha=0.35)
plt.axvline(-1.64, color="black", linestyle="--")

plt.title("Região de rejeição — Teste unilateral à esquerda (α = 0.05)")
plt.xlabel("z")
plt.ylabel("densidade")
plt.grid(alpha=0.3)
plt.show()

🟣 **Interpretação**

- Toda a área de rejeição está à esquerda.  
- Se a estatística de teste for menor que −1.64 → rejeitamos H₀.  

## 3.4 Teste unilateral à direita (right‑tailed)

🔵 **Conteúdo IBM**

> “Se a hipótese alternativa diz que a diferença é maior que zero,  
> rejeitamos H₀ quando a estatística T ou Z for maior que 1.64.”

Valor crítico:

$$
z > 1.64
$$

In [ ]:
plt.figure(figsize=(8, 3))
plt.plot(x, y, color="steelblue")

plt.fill_between(x, y, where=(x > 1.64), color="red", alpha=0.35)
plt.axvline(1.64, color="black", linestyle="--")

plt.title("Região de rejeição — Teste unilateral à direita (α = 0.05)")
plt.xlabel("z")
plt.ylabel("densidade")
plt.grid(alpha=0.3)
plt.show()

🟣 **Interpretação**

- Toda a área de rejeição está à direita.  
- Se a estatística de teste for maior que 1.64 → rejeitamos H₀.  

## 3.5 Conexão com o valor‑p

A regra geral:

- **Teste bilateral**  
  $$
  \text{valor‑p} = 2 \cdot P(Z > |z_{\text{obs}}|)
  $$

- **Teste unilateral à direita**  
  $$
  \text{valor‑p} = P(Z > z_{\text{obs}})
  $$

- **Teste unilateral à esquerda**  
  $$
  \text{valor‑p} = P(Z < z_{\text{obs}})
  $$

## 3.6 Exemplo IBM — Teste bilateral

O curso diz:

> “Se o valor absoluto da estatística for maior que 1.96, rejeitamos H₀.”

Vamos ilustrar.

In [ ]:
z_obs = 2.1
p_value = 2 * (1 - norm.cdf(abs(z_obs)))
z_obs, p_value

🟣 **Interpretação**

- |2.1| > 1.96 → rejeitamos H₀  
- valor‑p < 0.05  

Exatamente como no conteúdo IBM.

## 3.7 Exemplo IBM — Teste unilateral

O curso diz:

> “Se a estatística T ou Z for maior que 1.64, rejeitamos H₀ no teste unilateral.”

Vamos ilustrar.

In [ ]:
z_obs = 1.7
p_value = 1 - norm.cdf(z_obs)
z_obs, p_value

🟣 **Interpretação**

- 1.7 > 1.64 → rejeitamos H₀  
- valor‑p < 0.05  

Isso corresponde exatamente ao vídeo do módulo 5.

<details>
<summary>🟠 Explicação avançada — De onde vêm 1.96 e 1.64?</summary>

Esses valores são quantis da normal padrão:

- 1.96 é o quantil de 97.5%  
- 1.64 é o quantil de 95%  

Formalmente:

$$
\Phi(1.96) = 0.975
$$

$$
\Phi(1.64) = 0.95
$$

Onde $\Phi$ é a CDF da normal padrão.

</details>

<a id="t2samples"></a>
# 4. Teste T para duas amostras independentes

🔵 **Conteúdo oficial do módulo 5 (IBM)**

O curso explica que:

> “Um teste t é a comparação dos valores médios entre dois grupos.”

E apresenta o exemplo:
- média das avaliações de ensino de professoras: **3.90**, sd = **0.53**
- média das avaliações de ensino de professores: **4.06**, sd = **0.55**

O ponto central:

> “Ao conduzir um teste t, podemos assumir variâncias iguais ou desiguais.”

Vamos estruturar isso de forma clara e aprofundada.

## 4.1 O problema: comparar duas médias

Queremos testar:

$$
H_0: \mu_1 = \mu_2
$$

$$
H_a: \mu_1 \ne \mu_2
$$

Onde:
- grupo 1 = instrutores do sexo feminino  
- grupo 2 = instrutores do sexo masculino  

In [ ]:
female = ratings_df[ratings_df["gender"] == "female"]["eval"]
male   = ratings_df[ratings_df["gender"] == "male"]["eval"]

female.describe(), male.describe()

🟣 **Interpretação**

- As médias são próximas, mas não idênticas.  
- As variâncias são semelhantes, mas não iguais.  
- Precisamos decidir se assumimos variâncias iguais ou desiguais.  

## 4.2 Variâncias iguais vs desiguais

🔵 **Conteúdo IBM**

O curso diz:

> “Temos um teste t chamado teste de Levene para determinar a igualdade das variâncias.”

- Se o valor‑p do Levene < 0.05 → **variâncias desiguais**  
- Se o valor‑p ≥ 0.05 → **variâncias iguais**  

In [ ]:
stat, p_levene = levene(female, male, center="mean")
p_levene

🟣 **Interpretação**

- valor‑p ≈ 0.66  
- 0.66 > 0.05 → **não rejeitamos H₀**  
- portanto, **assumimos variâncias iguais**  

Isso está exatamente no conteúdo IBM.

## 4.3 Regra prática (IBM)

O curso também apresenta uma regra simples:

> “A razão entre a maior variância e a menor variância deve ser < 1.5  
> para assumir variâncias iguais.”

Vamos verificar.

In [ ]:
var_ratio = max(female.var(), male.var()) / min(female.var(), male.var())
var_ratio

🟣 **Interpretação**

- A razão é menor que 1.5  
- Portanto, **variâncias iguais** é uma suposição razoável  

## 4.4 Teste t com variâncias iguais (pooled)

Fórmula da variância combinada:

$$
s_p^2 =
\frac{(n_1 - 1)s_1^2 + (n_2 - 1)s_2^2}
{n_1 + n_2 - 2}
$$

Estatística t:

$$
t =
\frac{\bar{x}_1 - \bar{x}_2}
{s_p \sqrt{\frac{1}{n_1} + \frac{1}{n_2}}}
$$

Graus de liberdade:

$$
\nu = n_1 + n_2 - 2
$$

In [ ]:
from scipy.stats import ttest_ind

t_equal = ttest_ind(female, male, equal_var=True)
t_equal

🟣 **Interpretação**

- valor‑p > 0.05  
- **não rejeitamos H₀**  
- as médias não diferem estatisticamente  

Isso coincide com o conteúdo IBM.

## 4.5 Teste t com variâncias desiguais (Welch)

Mesmo que o Levene indique variâncias iguais, vamos comparar com Welch.

Welch não assume variâncias iguais:

$$
t =
\frac{\bar{x}_1 - \bar{x}_2}
{\sqrt{\frac{s_1^2}{n_1} + \frac{s_2^2}{n_2}}}
$$

Graus de liberdade (aproximados):

$$
\nu =
\frac{
\left(\frac{s_1^2}{n_1} + \frac{s_2^2}{n_2}\right)^2
}
{
\frac{s_1^4}{n_1^2(n_1 - 1)} +
\frac{s_2^4}{n_2^2(n_2 - 1)}
}
$$

In [ ]:
t_unequal = ttest_ind(female, male, equal_var=False)
t_unequal

🟣 **Interpretação**

- Welch também dá valor‑p > 0.05  
- Conclusão permanece: **não rejeitamos H₀**  

## 4.6 Visualização das médias com IC (t‑distribution)

In [ ]:
stats = ratings_df.groupby("gender")["eval"].agg(["mean", "std", "count"])

plt.figure(figsize=(7, 4))

for i, gender in enumerate(stats.index):
    mean = stats.loc[gender, "mean"]
    sd = stats.loc[gender, "std"]
    n = stats.loc[gender, "count"]
    se = sd / np.sqrt(n)
    
    df = n - 1
    t_crit = t.ppf(0.975, df)
    ci_low = mean - t_crit * se
    ci_high = mean + t_crit * se
    
    plt.errorbar(i, mean, yerr=[[mean-ci_low], [ci_high-mean]],
                 fmt="o", capsize=5, label=gender)

plt.xticks([0, 1], stats.index)
plt.ylabel("eval")
plt.title("Médias de avaliação por gênero com IC (t-distribution)")
plt.grid(alpha=0.3)
plt.show()

🟣 **Interpretação**

- Os intervalos de confiança se sobrepõem.  
- Isso sugere que as médias não diferem significativamente.  
- O teste t confirma isso.  

<details>
<summary>🟠 Explicação avançada — Por que Welch é mais robusto?</summary>

Welch ajusta os graus de liberdade para refletir variâncias diferentes.

Isso evita:
- inflar a taxa de erro tipo I,
- assumir homogeneidade quando ela não existe.

Em geral:
- Welch é mais seguro,  
- pooled é mais eficiente quando as variâncias são realmente iguais.

Por isso, muitos softwares usam Welch como padrão.

</details>

<a id="anova"></a>
# 5. ANOVA — Análise de Variância

🔵 **Conteúdo oficial do módulo 5 (IBM)**

O curso explica:

> “Quando queremos comparar médias de mais de dois grupos, usamos ANOVA.”

E apresenta o exemplo:
- Agrupar instrutores por idade  
- Comparar médias de avaliação (`eval`)  
- Comparar médias de beleza (`beauty`)  

A ANOVA usa a **distribuição F** para testar:

$$
H_0: \mu_1 = \mu_2 = \mu_3 = \dots
$$

$$
H_a: \text{Pelo menos uma média é diferente}
$$

## 5.1 Quando usar ANOVA?

Use ANOVA quando:

- há **3 ou mais grupos**  
- queremos comparar **médias**  
- as observações são independentes  
- as variâncias são aproximadamente iguais (homocedasticidade)  

ANOVA responde:

> “As médias dos grupos são estatisticamente iguais?”

## 5.2 A distribuição F

A estatística F é:

$$
F = \frac{\text{variabilidade entre grupos}}
         {\text{variabilidade dentro dos grupos}}
$$

Se as médias forem iguais:
- a variabilidade entre grupos é pequena  
- F tende a ser próximo de 1  

Se pelo menos uma média for diferente:
- a variabilidade entre grupos aumenta  
- F cresce  
- valor‑p diminui  

## 5.3 Exemplo IBM — Agrupando instrutores por idade

O curso cria três grupos:

- **Grupo 1:** idade ≤ 40  
- **Grupo 2:** 40 < idade ≤ 56.5  
- **Grupo 3:** idade ≥ 57  

Vamos reproduzir isso.

In [ ]:
# Criar grupos de idade
bins = [0, 40, 56.5, 100]
labels = ["≤40", "40–56.5", "≥57"]

ratings_df["age_group"] = pd.cut(ratings_df["age"], bins=bins, labels=labels, include_lowest=True)
ratings_df[["age", "age_group"]].head()

## 5.4 Estatísticas descritivas por grupo

In [ ]:
ratings_df.groupby("age_group")["eval"].agg(["mean", "std", "count"])

🟣 **Interpretação**

- As médias são próximas, mas não idênticas.  
- Precisamos verificar se essas diferenças são estatisticamente significativas.  

## 5.5 ANOVA para `eval` (exemplo IBM)

In [ ]:
group1 = ratings_df[ratings_df["age_group"] == "≤40"]["eval"]
group2 = ratings_df[ratings_df["age_group"] == "40–56.5"]["eval"]
group3 = ratings_df[ratings_df["age_group"] == "≥57"]["eval"]

f_stat, p_value = f_oneway(group1, group2, group3)
f_stat, p_value

🟣 **Interpretação**

- valor‑p ≈ 0.295  
- 0.295 > 0.05 → **não rejeitamos H₀**  
- As médias de `eval` **não diferem estatisticamente** entre os grupos de idade  

Isso coincide exatamente com o conteúdo IBM.

## 5.6 ANOVA para `beauty` (exemplo IBM)

O curso também testa se a pontuação de beleza difere por idade.

In [ ]:
group1_b = ratings_df[ratings_df["age_group"] == "≤40"]["beauty"]
group2_b = ratings_df[ratings_df["age_group"] == "40–56.5"]["beauty"]
group3_b = ratings_df[ratings_df["age_group"] == "≥57"]["beauty"]

f_stat_b, p_value_b = f_oneway(group1_b, group2_b, group3_b)
f_stat_b, p_value_b

🟣 **Interpretação**

- valor‑p ≈ 4.32 × 10⁻⁸  
- muito menor que 0.05  
- **rejeitamos H₀**  
- Pelo menos uma média de `beauty` difere entre os grupos  

Isso também coincide com o conteúdo IBM.

## 5.7 Visualização das médias por grupo

In [ ]:
stats = ratings_df.groupby("age_group")["eval"].agg(["mean", "std", "count"])

plt.figure(figsize=(7, 4))
plt.bar(stats.index, stats["mean"], color=["#4C72B0", "#55A868", "#C44E52"])
plt.ylabel("Média de eval")
plt.title("Média de avaliação por grupo de idade")
plt.grid(alpha=0.3)
plt.show()

🟣 **Interpretação**

- As médias são visualmente próximas.  
- Isso reforça o resultado da ANOVA: **não há diferença significativa**.  

## 5.8 Visualização da variabilidade (boxplot)

In [ ]:
plt.figure(figsize=(7, 4))
ratings_df.boxplot(column="eval", by="age_group")
plt.title("Distribuição de eval por grupo de idade")
plt.suptitle("")
plt.xlabel("Grupo de idade")
plt.ylabel("eval")
plt.grid(alpha=0.3)
plt.show()

🟣 **Interpretação**

- As distribuições são semelhantes.  
- Não há evidência visual de grandes diferenças.  

<details>
<summary>🟠 Explicação avançada — Derivação da estatística F</summary>

A ANOVA de um fator decompõe a variabilidade total:

$$
\text{SST} = \text{SSB} + \text{SSW}
$$

Onde:

- SST = soma total dos quadrados  
- SSB = soma dos quadrados entre grupos  
- SSW = soma dos quadrados dentro dos grupos  

A estatística F é:

$$
F = \frac{\text{SSB}/(k-1)}{\text{SSW}/(N-k)}
$$

Onde:
- k = número de grupos  
- N = tamanho total da amostra  

Se as médias forem iguais, F tende a 1.  
Se forem diferentes, F cresce.  

</details>

<a id="correlacao"></a>
# 6. Testes de Correlação

🔵 **Conteúdo oficial do módulo 5 (IBM)**

O curso explica que:

> “Se estivermos comparando duas variáveis categóricas, usamos o teste Qui‑quadrado.  
> Se estivermos comparando duas variáveis contínuas, usamos o teste de correlação de Pearson.”

Vamos seguir exatamente essa estrutura.

# 6.1 Correlação entre variáveis categóricas — Qui‑quadrado

Exemplo IBM:

- Variável 1: **gender** (male / female)  
- Variável 2: **tenure** (tenured / not tenured)  

Queremos testar:

$$
H_0: \text{gender e tenure são independentes}
$$

$$
H_a: \text{gender e tenure são associados}
$$

In [ ]:
# Tabela de contingência
contingency = pd.crosstab(ratings_df["tenure"], ratings_df["gender"])
contingency

## 6.1.1 Cálculo manual dos valores esperados (como no vídeo IBM)

O curso mostra como calcular:

$$
E_{ij} = \frac{(\text{total da linha}) \cdot (\text{total da coluna})}{\text{total geral}}
$$

In [ ]:
# Totais
row_totals = contingency.sum(axis=1)
col_totals = contingency.sum(axis=0)
grand_total = contingency.values.sum()

expected = np.outer(row_totals, col_totals) / grand_total
expected_df = pd.DataFrame(expected, index=contingency.index, columns=contingency.columns)
expected_df

🟣 **Interpretação**

- Estes são os valores que esperaríamos **se não houvesse associação**.  
- O teste Qui‑quadrado compara observado vs esperado.  

## 6.1.2 Cálculo do Qui‑quadrado (manual)

Fórmula:

$$
\chi^2 = \sum \frac{(O - E)^2}{E}
$$

In [ ]:
chi2_manual = ((contingency - expected_df)**2 / expected_df).to_numpy().sum()
chi2_manual

## 6.1.3 Teste Qui‑quadrado com SciPy

In [ ]:
chi2_stat, p_value, dof, expected_scipy = chi2_contingency(contingency)
chi2_stat, p_value, dof

🟣 **Interpretação**

- valor‑p ≈ 0.11  
- 0.11 > 0.05 → **não rejeitamos H₀**  
- Conclusão: **gender e tenure são independentes**  

Isso coincide exatamente com o conteúdo IBM.

## 6.1.4 Visualização da tabela de contingência

In [ ]:
contingency.plot(kind="bar", figsize=(7, 4))
plt.title("Distribuição de tenure por gênero")
plt.ylabel("Contagem")
plt.grid(alpha=0.3)
plt.show()

🟣 **Interpretação**

- A distribuição é semelhante entre os grupos.  
- Não há evidência visual de associação forte.  

<details>
<summary>🟠 Explicação avançada — Distribuição Qui‑quadrado</summary>

A estatística Qui‑quadrado segue:

$$
\chi^2_k = \sum_{i=1}^k Z_i^2
$$

onde $Z_i \sim N(0,1)$ independentes.

Propriedades:
- assimétrica  
- valores grandes → rejeição de H₀  
- depende dos graus de liberdade  

</details>

---
# 6.2 Correlação entre variáveis contínuas — Pearson

Exemplo IBM:

- Variável 1: **beauty**  
- Variável 2: **eval**  

Queremos testar:

$$
H_0: \rho = 0
$$

$$
H_a: \rho \ne 0
$$

In [ ]:
beauty = ratings_df["beauty"]
evals  = ratings_df["eval"]

pearson_r, p_value = pearsonr(beauty, evals)
pearson_r, p_value

🟣 **Interpretação**

- coeficiente r ≈ 0.18  
- valor‑p ≈ 4.25 × 10⁻⁵  
- valor‑p < 0.05 → **rejeitamos H₀**  
- Conclusão: existe correlação positiva entre beleza e avaliação  

Isso coincide exatamente com o conteúdo IBM.

## 6.2.1 Visualização — Scatter plot

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(beauty, evals, alpha=0.6)
plt.xlabel("beauty")
plt.ylabel("eval")
plt.title("Correlação entre beleza e avaliação")
plt.grid(alpha=0.3)
plt.show()

🟣 **Interpretação**

- A tendência é levemente crescente.  
- Isso corresponde ao r positivo encontrado.  

<details>
<summary>🟠 Explicação avançada — Fórmula do coeficiente de Pearson</summary>

O coeficiente r é:

$$
r = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}
{\sqrt{\sum (x_i - \bar{x})^2} \sqrt{\sum (y_i - \bar{y})^2}}
$$

Propriedades:
- r ∈ [−1, 1]  
- r = 1 → correlação perfeita positiva  
- r = −1 → correlação perfeita negativa  
- r = 0 → sem correlação linear  

O valor‑p testa se r é estatisticamente diferente de zero.  

</details>

<a id="aplicacoes"></a>
# 7. Aplicações com *teaching ratings*

🔵 **Conteúdo oficial do módulo 5 (IBM)**

O curso usa o dataset *teaching ratings* para aplicar:

- Teste T (duas amostras independentes)  
- Teste Z (quando aplicável)  
- ANOVA  
- Qui‑quadrado  
- Correlação de Pearson  

Aqui vamos integrar todos esses testes em perguntas reais.

---
# 7.1 Pergunta 1 — Professores homens recebem avaliações mais altas?

Hipóteses:

$$
H_0: \mu_{\text{male}} = \mu_{\text{female}}
$$

$$
H_a: \mu_{\text{male}} \ne \mu_{\text{female}}
$$

Este é um **teste bilateral**.

In [ ]:
female = ratings_df[ratings_df["gender"] == "female"]["eval"]
male   = ratings_df[ratings_df["gender"] == "male"]["eval"]

t_stat, p_value = ttest_ind(female, male, equal_var=True)
t_stat, p_value

🟣 **Interpretação**

- valor‑p > 0.05  
- **não rejeitamos H₀**  
- Não há evidência de diferença entre as médias  

Isso coincide com o conteúdo IBM.

---
# 7.2 Pergunta 2 — Professores mais jovens recebem notas diferentes?

Usamos ANOVA com os três grupos de idade criados anteriormente.

In [ ]:
f_stat, p_value = f_oneway(group1, group2, group3)
f_stat, p_value

🟣 **Interpretação**

- valor‑p ≈ 0.295  
- **não rejeitamos H₀**  
- As médias de `eval` não diferem entre os grupos de idade  

Exatamente como no vídeo IBM.

---
# 7.3 Pergunta 3 — A beleza do instrutor varia com a idade?

Usamos ANOVA para `beauty`.

In [ ]:
f_stat_b, p_value_b = f_oneway(group1_b, group2_b, group3_b)
f_stat_b, p_value_b

🟣 **Interpretação**

- valor‑p extremamente pequeno  
- **rejeitamos H₀**  
- Pelo menos uma média de `beauty` difere entre os grupos  

Isso também coincide com o conteúdo IBM.

---
# 7.4 Pergunta 4 — Gênero e estabilidade (tenure) são associados?

Variáveis categóricas → **Qui‑quadrado de independência**.

In [ ]:
contingency = pd.crosstab(ratings_df["tenure"], ratings_df["gender"])
chi2_stat, p_value, dof, expected = chi2_contingency(contingency)
chi2_stat, p_value

🟣 **Interpretação**

- valor‑p ≈ 0.11  
- **não rejeitamos H₀**  
- Gênero e estabilidade são independentes  

Exatamente como no conteúdo IBM.

---
# 7.5 Pergunta 5 — Beleza e avaliação estão correlacionadas?

Variáveis contínuas → **Correlação de Pearson**.

In [ ]:
pearson_r, p_value = pearsonr(ratings_df["beauty"], ratings_df["eval"])
pearson_r, p_value

🟣 **Interpretação**

- r ≈ 0.18 → correlação fraca positiva  
- valor‑p < 0.05 → estatisticamente significativa  
- Conclusão: existe relação entre beleza e avaliação  

Isso coincide com o vídeo IBM.

---
# 7.6 Pergunta 6 — Professores com mais alunos recebem notas diferentes?

Vamos dividir a variável `students` em dois grupos:

- turma pequena (≤ 40 alunos)  
- turma grande (> 40 alunos)  

In [ ]:
ratings_df["class_size"] = np.where(ratings_df["students"] <= 40, "small", "large")

small = ratings_df[ratings_df["class_size"] == "small"]["eval"]
large = ratings_df[ratings_df["class_size"] == "large"]["eval"]

ttest_ind(small, large, equal_var=False)

🟣 **Interpretação**

- valor‑p > 0.05  
- **não rejeitamos H₀**  
- Tamanho da turma não afeta significativamente a avaliação  

---
# 7.7 Pergunta 7 — Professores com maior beleza recebem mais alunos?

Pearson entre:
- beauty  
- students  

In [ ]:
pearsonr(ratings_df["beauty"], ratings_df["students"])

🟣 **Interpretação**

- r muito próximo de zero  
- valor‑p alto  
- **sem correlação** entre beleza e número de alunos  

---
# 7.8 Pergunta 8 — Professores mais bem avaliados recebem mais alunos?

Pearson entre:
- eval  
- students  

In [ ]:
pearsonr(ratings_df["eval"], ratings_df["students"])

🟣 **Interpretação**

- r ≈ 0.05  
- valor‑p alto  
- **sem correlação** entre avaliação e tamanho da turma  

---
# 7.9 Visualização integrada — Matriz de correlação

In [ ]:
plt.figure(figsize=(6, 5))
corr = ratings_df[["eval", "beauty", "age", "students"]].corr()
plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar()
plt.xticks(range(len(corr)), corr.columns)
plt.yticks(range(len(corr)), corr.columns)
plt.title("Matriz de correlação")
plt.show()

🟣 **Interpretação**

- A única correlação notável é beauty × eval  
- As demais são fracas ou inexistentes  

<details>
<summary>🟠 Explicação avançada — Por que integrar vários testes?</summary>

Em estatística aplicada:

- raramente uma única técnica responde tudo  
- testes complementares revelam padrões diferentes  
- ANOVA, T‑test, Qui‑quadrado e Pearson analisam aspectos distintos  

O dataset teaching ratings é perfeito para demonstrar isso:

- médias por gênero → T‑test  
- médias por idade → ANOVA  
- associação categórica → Qui‑quadrado  
- relação contínua → Pearson  

</details>

<a id="exercicios"></a>
# 8. Exercícios Guiados

Estes exercícios consolidam todo o conteúdo do módulo 5:

- Teste Z  
- Teste T (variâncias iguais e desiguais)  
- ANOVA  
- Qui‑quadrado  
- Correlação de Pearson  
- Interpretação de hipóteses  
- Regiões de rejeição  

Cada exercício foi desenhado para reforçar a teoria e a prática.

---
# 📝 Exercício 1 — Teste Z (média vs valor hipotético)

Suponha que a média populacional de avaliações seja **4.0** e que o desvio padrão
populacional seja **0.5**.

Uma amostra de **n = 40** instrutores tem média **4.12**.

Teste:

$$
H_0: \mu = 4.0
$$

$$
H_a: \mu > 4.0
$$

Use **teste Z**.

1. Calcule a estatística Z.  
2. Calcule o valor‑p.  
3. Conclua ao nível α = 0.05.

In [ ]:
# TODO: seu código aqui

---
# 📝 Exercício 2 — Teste T (média vs valor hipotético)

Agora suponha que o desvio padrão populacional **não é conhecido**.

Uma amostra de **n = 15** instrutores tem:
- média = 3.85  
- desvio padrão amostral = 0.62  

Teste:

$$
H_0: \mu = 4.0
$$

$$
H_a: \mu \ne 4.0
$$

Use **teste T bilateral**.

In [ ]:
# TODO: seu código aqui

---
# 📝 Exercício 3 — Teste T para duas amostras independentes

Usando o dataset teaching ratings:

Teste se instrutores **minoritários** têm média de avaliação diferente de instrutores **não minoritários**.

1. Separe os grupos.  
2. Teste variâncias iguais vs desiguais (Levene).  
3. Execute o teste T apropriado.  
4. Interprete o valor‑p.

In [ ]:
# TODO: seu código aqui

---
# 📝 Exercício 4 — ANOVA

Use a variável `age_group` criada anteriormente.

Teste:

$$
H_0: \mu_{\le 40} = \mu_{40–56.5} = \mu_{\ge 57}
$$

$$
H_a: \text{Pelo menos uma média é diferente}
$$

1. Execute ANOVA para `eval`.  
2. Execute ANOVA para `beauty`.  
3. Compare os valores‑p.  
4. Interprete.

In [ ]:
# TODO: seu código aqui

---
# 📝 Exercício 5 — Qui‑quadrado de independência

Teste se existe associação entre:

- `gender`  
- `tenure`  

1. Crie a tabela de contingência.  
2. Calcule o Qui‑quadrado.  
3. Interprete o valor‑p.  
4. Compare com o exemplo do módulo 5.

In [ ]:
# TODO: seu código aqui

---
# 📝 Exercício 6 — Correlação de Pearson

Teste se existe correlação entre:

- `age`  
- `eval`  

1. Calcule o coeficiente r.  
2. Calcule o valor‑p.  
3. Plote o scatter plot.  
4. Interprete.

In [ ]:
# TODO: seu código aqui

---
# 📝 Exercício 7 — Regiões de rejeição

Para cada caso, determine se a estatística de teste leva à rejeição de H₀.

1. Teste bilateral, z = 2.15  
2. Teste unilateral à direita, z = 1.55  
3. Teste unilateral à esquerda, z = −1.80  

Use α = 0.05.

In [ ]:
# TODO: escreva suas conclusões aqui

---
# 📝 Exercício 8 — Visualização

Plote a curva normal padrão e destaque:

- região crítica bilateral (±1.96)  
- região crítica unilateral direita (1.64)  
- região crítica unilateral esquerda (−1.64)  

Use cores diferentes.

In [ ]:
# TODO: seu código aqui

---
# 📝 Exercício 9 — Interpretação conceitual

Responda em texto:

1. O que significa rejeitar H₀?  
2. O que significa não rejeitar H₀?  
3. Qual a diferença entre valor‑p e valor crítico?  
4. Por que ANOVA usa a distribuição F?  
5. Por que o teste de Pearson detecta apenas correlação linear?  

Escreva respostas claras e concisas.

In [ ]:
# TODO: escreva suas respostas aqui (em markdown)

---
# 📝 Exercício 10 — Integração final

Escolha **qualquer duas variáveis** do dataset teaching ratings e:

1. Determine se são categóricas ou contínuas.  
2. Escolha o teste apropriado (T, Z, ANOVA, Qui‑quadrado, Pearson).  
3. Execute o teste.  
4. Interprete o resultado.  
5. Explique por que o teste escolhido é o correto.  

Este exercício integra todo o módulo.

In [ ]:
# TODO: seu código aqui

<a id="apendice"></a>
# 9. Apêndice Matemático Avançado

Esta seção é opcional e serve como referência teórica para:

- testes Z e T  
- distribuição t  
- distribuição F  
- ANOVA  
- Qui‑quadrado  
- correlação de Pearson  
- graus de liberdade  
- valores críticos  

Use este apêndice como consulta rápida ou aprofundamento.

---
# 9.1 Derivação da estatística Z

Quando o desvio padrão populacional é conhecido:

$$
Z = \frac{\bar{X} - \mu_0}{\sigma / \sqrt{n}}
$$

Isso vem diretamente da padronização da normal:

$$
\bar{X} \sim N\left(\mu, \frac{\sigma^2}{n}\right)
$$

Logo:

$$
\frac{\bar{X} - \mu}{\sigma/\sqrt{n}} \sim N(0,1)
$$

---
# 9.2 Derivação da estatística T

Quando σ é desconhecido, substituímos por s:

$$
T = \frac{\bar{X} - \mu_0}{s / \sqrt{n}}
$$

A distribuição resultante é:

$$
T \sim t_{n-1}
$$

<details>
<summary>🟠 Demonstração — Por que T segue a distribuição t?</summary>

Se:

- $Z \sim N(0,1)$  
- $V \sim \chi^2_\nu$  
- $Z$ e $V$ independentes  

então:

$$
T = \frac{Z}{\sqrt{V/\nu}}
$$

segue uma distribuição t com $\nu$ graus de liberdade.

A variabilidade extra no denominador (estimativa de σ) gera as caudas mais pesadas.

</details>

---
# 9.3 Graus de liberdade (df)

Em estatística, graus de liberdade representam:

> “O número de valores independentes que podem variar.”

Exemplos:

- Para o teste T de uma amostra:  
  $$
  df = n - 1
  $$

- Para T de duas amostras (variâncias iguais):  
  $$
  df = n_1 + n_2 - 2
  $$

- Para Welch (variâncias desiguais):  
  fórmula aproximada (complexa)

- Para ANOVA:  
  - entre grupos: $df_1 = k - 1$  
  - dentro dos grupos: $df_2 = N - k$  

---
# 9.4 Distribuição F (ANOVA)

A estatística F é:

$$
F = \frac{\text{MSB}}{\text{MSW}}
$$

Onde:

- MSB = variância entre grupos  
- MSW = variância dentro dos grupos  

A distribuição F é assimétrica e depende de dois graus de liberdade:

$$
F \sim F(df_1, df_2)
$$

<details>
<summary>🟠 Demonstração — Decomposição da variância na ANOVA</summary>

A soma total dos quadrados é:

$$
SST = \sum_{i=1}^N (x_i - \bar{x})^2
$$

Ela se decompõe em:

$$
SST = SSB + SSW
$$

Onde:

- SSB = soma dos quadrados entre grupos  
- SSW = soma dos quadrados dentro dos grupos  

A razão entre variâncias gera a estatística F.

</details>

---
# 9.5 Distribuição Qui‑quadrado

A estatística Qui‑quadrado é:

$$
\chi^2 = \sum \frac{(O - E)^2}{E}
$$

Ela segue:

$$
\chi^2_k = \sum_{i=1}^k Z_i^2
$$

onde $Z_i \sim N(0,1)$ independentes.

---
# 9.6 Correlação de Pearson

O coeficiente r é:

$$
r = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}
{\sqrt{\sum (x_i - \bar{x})^2} \sqrt{\sum (y_i - \bar{y})^2}}
$$

O valor‑p testa:

$$
H_0: \rho = 0
$$

<details>
<summary>🟠 Demonstração — Distribuição de r sob H₀</summary>

Se $X$ e $Y$ são normais bivariados com $\rho = 0$:

$$
t = r \sqrt{\frac{n-2}{1-r^2}}
$$

segue:

$$
t \sim t_{n-2}
$$

Isso permite calcular o valor‑p.

</details>

---
# 9.7 Valores críticos (1.96 e 1.64)

Para α = 0.05:

- Teste bilateral:  
  $$
  z_{0.975} = 1.96
  $$

- Teste unilateral:  
  $$
  z_{0.95} = 1.64
  $$

---
# 9.8 Erros tipo I e tipo II

- **Erro tipo I (α):** rejeitar H₀ quando ela é verdadeira  
- **Erro tipo II (β):** não rejeitar H₀ quando ela é falsa  

A potência do teste é:

$$
1 - \beta
$$

---
# 9.9 Teorema Central do Limite (TCL)

O TCL garante que:

$$
\frac{\bar{X} - \mu}{\sigma/\sqrt{n}} \to N(0,1)
$$

conforme $n \to \infty$.

<details>
<summary>🟠 Demonstração — via funções características (esboço)</summary>

1. A função característica da soma é o produto das funções características.  
2. Expandindo em série de Taylor, o termo dominante é quadrático.  
3. Isso leva à forma exponencial da normal.  

</details>

---
# 9.10 Lei dos Grandes Números (LLN)

A média amostral converge para a média populacional:

$$
\bar{X}_n \to \mu
$$

quando $n \to \infty$.

---
# 9.11 Resumo visual das distribuições

In [ ]:
x = np.linspace(-4, 4, 400)

plt.figure(figsize=(8, 4))
plt.plot(x, norm.pdf(x), label="Normal", linewidth=2)
plt.plot(x, t.pdf(x, df=5), label="t (df=5)")
plt.plot(x, t.pdf(x, df=30), label="t (df=30)")
plt.title("Comparação: Normal vs t-distribution")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

🟣 **Interpretação**

- df pequeno → caudas mais pesadas  
- df grande → aproxima a normal  